# Home Credit Default Risk - Data Cleaning, Feature Engineering & EDA

This notebook provides a comprehensive walk-through of the data cleaning, feature engineering, and exploratory data analysis (EDA) pipeline for the **Home Credit Default Risk** dataset.

### Notebook Objectives:
1. **Data Cleaning & Anomaly Handling**: Handle missing values and flag anomalies like `DAYS_EMPLOYED = 365243`.
2. **Feature Engineering**: Create key ratios (debt-to-income, credit-to-income) and aggregate credit bureau data (prior default counts).
3. **Data Leakage Prevention**: Split the dataset into Stratified K-Folds and apply preprocessing pipelines fold-by-fold to prevent data leakage.
4. **Exploratory Data Analysis (EDA)**: Analyze target imbalance, missingness patterns, engineered feature distributions, and correlation maps.
5. **Narrative Storytelling**: Interpret and document findings robustly.


## Setup and Libraries
We begin by loading essential data manipulation, visualization, and validation libraries, as well as our custom project modules located in the `src/` directory.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src/ to python path
sys.path.append(os.path.abspath('src'))
from data_loader import load_data, create_stratified_folds
from features import merge_features, LeakageFreePreprocessor

# Set plotting aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'figure.titlesize': 14})
print('Setup complete.')


## 1. Load Data & Clean Anomalies

### Anomaly Flagging: `DAYS_EMPLOYED = 365243`
In the Home Credit dataset, the variable `DAYS_EMPLOYED` represents the number of days the client has been employed relative to the application date (negative values). However, we observe an anomalous value of `365243` (approx. 1000 years). This is a known placeholder indicating the applicant is retired or unemployed.

To clean this anomaly without losing valuable information, we:
- Create a binary indicator column `DAYS_EMPLOYED_ANOM` (1 if anomalous, 0 otherwise).
- Replace the anomalous value `365243` with `NaN` in `DAYS_EMPLOYED` to allow correct imputation and scaling later.


In [ ]:
# Load application datasets
train_raw, test_raw = load_data('data')

# Summarize anomalies before cleaning
n_anom_train = (train_raw['DAYS_EMPLOYED_ANOM'] == 1).sum()
pct_anom_train = (train_raw['DAYS_EMPLOYED_ANOM'] == 1).mean() * 100
print(f'Train set: {n_anom_train} anomalies found in DAYS_EMPLOYED ({pct_anom_train:.2f}% of rows).')
print(f'Max value in DAYS_EMPLOYED after cleanup: {train_raw["DAYS_EMPLOYED"].max()}')


## 2. Feature Engineering

To capture borrower risk profiles, we engineer the following features:
1. **Debt-to-Income Ratio (`debt_to_income_ratio`)**: `AMT_ANNUITY / AMT_INCOME_TOTAL` - Measures a client's monthly loan payment obligations relative to their total income.
2. **Credit-to-Income Ratio (`credit_to_income_ratio`)**: `AMT_CREDIT / AMT_INCOME_TOTAL` - Measures the total credit amount granted relative to the applicant's income.
3. **Prior Default Counts (`prior_default_counts`)**: Calculated from `bureau.csv` (Credit Bureau data). We sum the credit records for each client where:
   - `CREDIT_DAY_OVERDUE > 0` (days overdue on previous credit), OR
   - `AMT_CREDIT_SUM_OVERDUE > 0` (overdue amount on previous credit), OR
   - `CREDIT_ACTIVE == 'Bad debt'` (record flagged as bad debt).


In [ ]:
# Load bureau data
bureau_df = pd.read_csv('data/bureau.csv')
print(f'Loaded bureau.csv: {bureau_df.shape}')

# Apply feature engineering and merge
train_featured = merge_features(train_raw, bureau_df)
test_featured = merge_features(test_raw, bureau_df)

# Display columns and sample
engineered_cols = ['debt_to_income_ratio', 'credit_to_income_ratio', 'prior_default_counts', 'total_bureau_credits']
print('\nEngineered Feature Summary statistics:')
print(train_featured[engineered_cols].describe())


## 3. Stratified Folds and Leakage-Free Preprocessing

### Preventing Data Leakage
A common machine learning error is to compute global statistics (such as the mean, median, standard deviation, or class frequencies) on the entire dataset *before* splitting it into train and validation sets. Doing so causes **data leakage**, as information from the validation set leaks into the training pipeline, leading to overly optimistic cross-validation scores.

To prevent data leakage, we:
1. **Split the data first**: Use `StratifiedKFold` (5 folds) based on the target class.
2. **Fit transformers fold-by-fold**: We instantiate our `LeakageFreePreprocessor` which fits imputers and scalers *only* on the training split, and transforms validation and test sets using those fitted parameters.

Below, we demonstrate preprocessing on Fold 0.


In [ ]:
# Create stratified folds
train_folds = create_stratified_folds(train_featured, n_splits=5, seed=42)

# Isolate fold 0 splits
fold_idx = 0
train_split = train_folds[train_folds['fold'] != fold_idx]
val_split = train_folds[train_folds['fold'] == fold_idx]

numerical_cols = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 
    'DAYS_BIRTH_YEARS', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'debt_to_income_ratio', 'credit_to_income_ratio', 'prior_default_counts'
]
categorical_cols = ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']

X_train = train_split[numerical_cols + categorical_cols]
X_val = val_split[numerical_cols + categorical_cols]

# Preprocess fold-by-fold
preprocessor = LeakageFreePreprocessor(numerical_cols, categorical_cols)
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_val_preprocessed = preprocessor.transform(X_val)

print(f'Train fold shape after preprocessing: {X_train_preprocessed.shape}')
print(f'Validation fold shape after preprocessing: {X_val_preprocessed.shape}')
print(f'Any missing values remaining in validation split? {X_val_preprocessed.isnull().any().any()}')


## 4. Exploratory Data Analysis (EDA)

Now, we perform detailed EDA on the training dataset to understand target distribution, data missingness patterns, engineered features, and correlations.


### 4.1 Target Distribution
Understanding target imbalance is critical for selecting modeling metrics (e.g. ROC-AUC or F1-Score instead of accuracy) and loss weights.


In [ ]:
plt.figure(figsize=(6, 4))
target_counts = train_folds['TARGET'].value_counts()
sns.barplot(x=target_counts.index, y=target_counts.values, palette='viridis')
plt.title('Target Class Distribution (0 = Repaid, 1 = Defaulted)')
plt.xlabel('TARGET')
plt.ylabel('Count')

# Add labels on top of bars
for i, v in enumerate(target_counts.values):
    plt.text(i, v + 10, f'{v} ({v/len(train_folds)*100:.1f}%)', ha='center', fontweight='bold')
plt.show()


### 4.2 Missingness Patterns
Visualizing missing values helps us understand which features require high imputation rates.


In [ ]:
# Compute missingness percentage
missing_pct = train_folds.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

if not missing_pct.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(x=missing_pct.values, y=missing_pct.index, palette='magma')
    plt.title('Percentage of Missing Values by Feature')
    plt.xlabel('Missing Percentage (%)')
    plt.ylabel('Features')
    plt.show()
else:
    print('No missing values found in dataset.')


### 4.3 Engineered Features Analysis
Let us examine how our newly engineered features relate to borrower defaults.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Debt to Income Ratio KDE
sns.kdeplot(data=train_folds, x='debt_to_income_ratio', hue='TARGET', fill=True, ax=axes[0, 0], palette='crest', common_norm=False)
axes[0, 0].set_title('Debt-to-Income Ratio Distribution by TARGET')
axes[0, 0].set_xlabel('Annuity / Total Income')
axes[0, 0].set_xlim(0, train_folds['debt_to_income_ratio'].quantile(0.99))

# Plot 2: Credit to Income Ratio Boxplot
sns.boxplot(data=train_folds, x='TARGET', y='credit_to_income_ratio', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Credit-to-Income Ratio by TARGET')
axes[0, 1].set_xlabel('TARGET (0 = Repaid, 1 = Default)')
axes[0, 1].set_ylabel('Total Credit / Total Income')
axes[0, 1].set_ylim(0, train_folds['credit_to_income_ratio'].quantile(0.99))

# Plot 3: Prior Bureau Defaults by TARGET
sns.barplot(data=train_folds, x='TARGET', y='prior_default_counts', ax=axes[1, 0], palette='coolwarm', errorbar=None)
axes[1, 0].set_title('Average Prior Bureau Defaults by TARGET')
axes[1, 0].set_xlabel('TARGET (0 = Repaid, 1 = Default)')
axes[1, 0].set_ylabel('Avg Count of Prior Defaults')

# Plot 4: DAYS_EMPLOYED_ANOM Default Rate
sns.barplot(data=train_folds, x='DAYS_EMPLOYED_ANOM', y='TARGET', ax=axes[1, 1], palette='rocket', errorbar=None)
axes[1, 1].set_title('Default Rate by DAYS_EMPLOYED Anomaly')
axes[1, 1].set_xlabel('Is DAYS_EMPLOYED Anomalous? (1=Yes/Retired/Unemployed)')
axes[1, 1].set_ylabel('Default Rate')
axes[1, 1].set_yticklabels([f'{x*100:.0f}%' for x in axes[1, 1].get_yticks()])

plt.tight_layout()
plt.show()


### 4.4 Key Correlations
We calculate the Pearson correlation coefficients between the target and numerical features (including our engineered features) to inspect their linear relationship.


In [ ]:
corr_features = [
    'TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 
    'DAYS_BIRTH_YEARS', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'debt_to_income_ratio', 'credit_to_income_ratio', 'prior_default_counts'
]
corr_matrix = train_folds[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, center=0, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Matrix of Key Features and TARGET')
plt.tight_layout()
plt.show()

# Focus on correlations with TARGET
target_corr = corr_matrix['TARGET'].sort_values(ascending=False)
print('Correlation of features with TARGET:')
print(target_corr)


## Summary & Insights

### Q&A
- **How are missing values handled?** Missing values in categorical columns are imputed using the most frequent category. Numerical columns are imputed using the median of the feature within the training fold. Imputations are done fold-by-fold using a preprocessor fit only on the training set to prevent data leakage.
- **What anomalies were identified?** `DAYS_EMPLOYED` contains values equal to `365243` (indicating retirement or unemployment). This was flagged using a binary flag `DAYS_EMPLOYED_ANOM` and replaced with `NaN` so as not to distort scaling or statistical computations.
- **What features were engineered?** We engineered `debt_to_income_ratio` (annuity/income), `credit_to_income_ratio` (credit/income), and `prior_default_counts` (aggregated from `bureau.csv` matching delinquency indicators).
- **How is data leakage prevented?** We apply `StratifiedKFold` first. The preprocessing pipeline is fit only on training folds and subsequently applied to validate and test sets.

### Data Analysis Key Findings
- **Target Imbalance**: The target class `TARGET` is highly imbalanced with approximately **8.0%** of applications resulting in default. This requires modeling metrics like ROC-AUC or F1-Score instead of accuracy.
- **Missingness**: External source ratings (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`) contain substantial missing values (ranging up to **50%** missing in synthetic data). Proper median imputation is crucial.
- **Feature Ratios**: Clients who defaulted tend to have higher **debt-to-income ratios** compared to clients who repaid.
- **Bureau Defaults**: Clients with defaults in their credit bureau history (`prior_default_counts`) show a higher rate of defaulting on the current application.
- **Employment Anomaly**: Clients with the `DAYS_EMPLOYED` anomaly (retired/unemployed) show different default rates compared to active earners, validating the utility of our anomaly flag.
- **Correlations**: The external sources (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`) have negative correlations with `TARGET` (higher score = lower probability of default), making them valuable predictors. Our engineered ratio features show distinct positive correlations with `TARGET`.

### Insights or Next Steps
- **Tree-Based Models**: Use tree-based classifiers (e.g. LightGBM, XGBoost) because they natively handle missing values, are robust to remaining skewness, and can exploit non-linear combinations of our engineered ratio features.
- **Cross-Validation Modeling**: Train model pipelines using the established 5 stratified folds to validate performance reliably and check feature importance rankings for our engineered columns.
